# 02 - Quality checks
Results of the 14 rules in `sql/03_checks.sql`, and the effect of the plausibility filter (QC14).

In [1]:
import pandas as pd, duckdb
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
M, C = ROOT / "data" / "marts", ROOT / "data" / "clean"
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 20)

In [2]:
pd.read_csv(C / 'quality_summary.csv')

,check_id,check_name,severity,rows_flagged,materials_affected
0,QC01,Missing month,error,7,2
1,QC02,Value without quantity,warn,6006,15
2,QC03,Unit-value outlier,warn,49,11
3,QC04,Thin trade month,info,24,5
4,QC05,Nomenclature change,info,4,2
5,QC08,Inconsistent units (kg vs m3),warn,47,2
6,QC09,Partner country not disclosed,info,154,8
7,QC11,Stale source,info,6,0
8,QC13,Estimated quantity (Comtrade),info,56,0
9,QC14,Implausible partner flow (excluded),warn,6712,15


In [3]:
qc = pd.read_csv(C / 'quality_checks.csv')
qc[qc.check_id == 'QC03'].sort_values('value', key=abs, ascending=False).head(10)[['material_id', 'period', 'value', 'detail']]

,material_id,period,value,detail
6036,HE,2026-08-01,12.6,unit value 79.89 EUR/kg is 213.0% away from it...
6019,GE,2023-09-01,8.1,unit value 5725.13 EUR/kg is 390.0% away from ...
6035,HE,2026-07-01,5.5,unit value 44.07 EUR/kg is 64.0% away from its...
6044,NE,2024-06-01,-5.5,unit value 7.25 EUR/kg is -96.0% away from its...
6016,GE,2018-07-01,5.4,unit value 2744.29 EUR/kg is 190.0% away from ...
6052,POLYSI,2024-07-01,-5.4,unit value 7.7 EUR/kg is -75.0% away from its ...
6042,NE,2022-05-01,-5.3,unit value 21.45 EUR/kg is -95.0% away from it...
6048,NH3,2023-08-01,-5.2,unit value 0.3 EUR/kg is -41.0% away from its ...
6020,GE,2025-05-01,-5.1,unit value 796.91 EUR/kg is -63.0% away from i...
6056,POLYSI,2025-10-01,-5.1,unit value 3.29 EUR/kg is -72.0% away from its...


In [4]:
m = pd.read_csv(M / 'monthly_prices.csv')
ne = m[m.material_id == 'NE'][['period', 'eu_import_uv_raw_eur_kg', 'eu_import_uv_eur_kg']]
ne.tail(12).round(1)

,period,eu_import_uv_raw_eur_kg,eu_import_uv_eur_kg
1125,2025-09-01,198.3,198.1
1126,2025-10-01,408.7,601.1
1127,2025-11-01,51.5,246.4
1128,2025-12-01,39.6,66.9
1129,2026-01-01,231.6,248.7
1130,2026-02-01,89.4,89.4
1131,2026-03-01,115.4,304.8
1132,2026-04-01,107.8,119.1
1133,2026-05-01,429.2,490.9
1134,2026-06-01,63.3,65.0
